# MultiAgent

### · Subagent 一个主agent，多个子agent，子agent将任务返回给主agent，由主agent统合返回
### · Handoffs 工作流形式，基于langgraph
### · Skills 只有一个agent但可以加载多个skill
### · Router 有一个Router agent，根据输入选择子agent执行任务

## Weeding Agent

### 环境准备

In [1]:
### 环境准备
import sys
import asyncio
import io
from dotenv import load_dotenv

load_dotenv()

if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    # 2. Redirect stderr BEFORE importing any MCP module:
    # mcp.client.stdio captures sys.stderr as the default errlog at import time,
    # and notebook consoles wrap sys.stderr without a real fileno()
    try:
        sys.stderr.fileno()
    except (io.UnsupportedOperation, AttributeError):
        sys.stderr = sys.__stderr__

from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_community.utilities import SQLDatabase
from langchain.agents import AgentState
from langchain_tavily import TavilySearch
from langchain.tools import tool
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage, ToolMessage, AIMessage
from langgraph.types import Command
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

### 航班agent

In [2]:
flight_client = MultiServerMCPClient(
    {
        "time" : {
            "transport" : "stdio",
            "command" : "uvx",
            "args" : ["mcp-server-time"]
        },
        "kiwi-flight-search" : {
            "transport" : "http",
            "url" : "https://mcp.kiwi.com"
        }
}
)

tools = await flight_client.get_tools()

In [3]:
flight_agent = create_agent(
    model="deepseek-flash",
    tools=tools,
    system_prompt="""
    你是一个旅行助手Agent。你为用户搜索到达婚礼地点的理想航班，您不能再追问任何后续问题。
    您必须根据以下标准找到最佳航班选择：
    - 价格（最低，经济舱）
    - 持续时间（最短）
    - 日期（您认为在该地点举行婚礼的最佳时间）
    为了使事情变得简单，只寻找一张票，单程。
    你不会得到额外的信息，只有出发地和目的地。你的工作是批判性地思考最佳选择。
    一旦你找到了最好的选项，让用户知道你的候选列表。
    调用搜索航班工具时，如果是地点在中国，需要使用zh-cn作为Locale参数。
    """
)

#### 测试

In [9]:
response = await flight_agent.ainvoke({"messages" : [HumanMessage("帮我看看明天昆明到上海的航班")]})

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

帮我看看明天昆明到上海的航班
================================== Ai Message ==================================
Tool Calls:
  get_current_time (call_00_RQwJKbSoUNzrOhPEULBX9052)
 Call ID: call_00_RQwJKbSoUNzrOhPEULBX9052
  Args:
    timezone: Asia/Shanghai
================================= Tool Message =================================
Name: get_current_time

[{'type': 'text', 'text': '{\n  "timezone": "Asia/Shanghai",\n  "datetime": "2026-11-21T12:28:23+08:00",\n  "day_of_week": "Saturday",\n  "is_dst": false\n}', 'id': 'lc_0197f1f2-def8-49cd-8f50-903dc1b24f87'}]
================================== Ai Message ==================================
Tool Calls:
  search-flight (call_00_4FVPcrfw04P5LVAbY0Y65577)
 Call ID: call_00_4FVPcrfw04P5LVAbY0Y65577
  Args:
    flyFrom: 昆明
    flyTo: 上海
    departureDate: 22/11/2026
    cabinClass: M
    adults: 1
    currency: CNY
    locale: zh-cn
    sort: price
========================

### 场地agent

In [4]:
web_search = TavilySearch(
    max_results=5,
    topic="general"
)

In [5]:
venue_agent = create_agent(
    model=ChatOllama(model="qwen2.5:7b"),
    tools=[web_search],
    system_prompt="""
    你是场地专家，你在指定的位置搜索满足要求的场地。
    你不能再追问任何后续问题，你必须根据以下标准找到最佳的场地选择：
    - 价格（最低）
    - 容量（完全匹配）
    - 评价（最高）
    您可能需要进行多次搜索以迭代地找到最佳选项。
    """
)

#### 测试

In [4]:
response = await venue_agent.ainvoke({"messages" : [HumanMessage("帮我找一个在上海的场地")]})
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

帮我找一个在上海的场地
================================== Ai Message ==================================

为了找到合适的场地，我将首先进行初步搜索，筛选出上海地区的场地，并关注价格最低、容量完全匹配以及评价最高的选项。我们将从价格最低的角度开始搜索。

首先，我将执行一个基本的搜索，以获取一些初步的候选场地信息。
Tool Calls:
  tavily_search (fd4b8056-7f19-403e-bd30-e48fcfd15a6d)
 Call ID: fd4b8056-7f19-403e-bd30-e48fcfd15a6d
  Args:
    query: 上海 场地 租赁
    search_depth: basic
================================= Tool Message =================================
Name: tavily_search

{"query": "上海 场地 租赁", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.thestorefront.com/hk/search/shanghai/event", "title": "上海活動空間出租 | Storefront", "content": "很难得出一个单一的答案，因为它取决于许多因素，例如地点、活动日期、定价类型等等。然而，我们发现上海更独特的一个特点是活动场地的租赁成本往往非常零散，这完全取决于您是否愿意挥霍。 上海的最低消费价格从每天 2,000 日元起，对于更高档、更壮观的空间，尤其是在新华区，可以高达 72,000 日元。 对于公司活动，如会议或商务会议，预计每天支付 3,000 日元起，具体取决于场地的独特或优雅程度。在组织公司活动时，最好的选择是提供代表费的场地，其中包括场地租用费、午餐、茶点和您需要的所有会议

### 音乐agent

#### 音乐数据准备

In [5]:
import sqlite3

# 提取出来的数据 (歌名, 歌手, 专辑, 流派/标签, 时长秒数)
music_data = [
    ("逐星河 (Instrumental)", "铁痕电台-MSR / 原田萌喜", "逐星河", "超清母带", 214),
    ("牵丝戏 (笛子纯音乐)", "无人区 / 吉泽树", "牵丝戏", "超清母带", 122),
    ("我不曾忘记 (游戏《原神》2023新春会同人曲)", "花玲 / 张安琪 / 沐霏", "我不曾忘记", "超清母带", 230),
    ("某一夜的天河", "铁痕电台-MSR", "离离剑上尘", "超清母带", 128),
    ("Alive (TV动画《明日方舟：黎明前奏》片头曲)", "ReoNa", "Alive", "超清母带", 258),
    ("BRE@TH//LESS (TV动画《ALDNOAH.ZERO》插曲)", "小林未郁", "アルドノア・ゼロ オリジナル・サウンドトラック", "超清母带", 243),
    ("Introspection", "Waking Dreams", "Introspection", "沉浸声", 324),
    ("Kami no michi (神の道)", "Sprnova", "Kami no michi", "沉浸声", 167),
    ("This Place Is a Shelter (容身之处)", "Olafur Arnalds", "Living Room Songs", "超清母带", 138),
    ("BrooklynBlood (Phonk) (Sped)", "ZsFlovexl / ZzzzHAO", "BrooklynBlood (Phonk)", "超清母带", 111),
    ("疾行锐进 Swift Advance", "HOYO-MiX", "原神-镌有慈怜之垠 Where Mercy Endures", "沉浸声", 96),
    ("砺剑峥嵘", "铁痕电台-MSR", "向渊行OST", "超清母带", 121),
    ("終焉の光", "八時八分", "終焉之光", "超清母带", 215),
    ("Find Me", "*Luna", "Find Me", "超清母带", 192)
]

def init_db_and_import():
    conn = sqlite3.connect('music.db')
    cursor = conn.cursor()

    # 建表
    cursor.executescript('''
    CREATE TABLE IF NOT EXISTS Artists (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name VARCHAR(100) NOT NULL UNIQUE
    );
    CREATE TABLE IF NOT EXISTS Albums (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name VARCHAR(100) NOT NULL,
        artist_id INTEGER NOT NULL,
        year_published INTEGER,
        FOREIGN KEY (artist_id) REFERENCES Artists(id)
    );
    CREATE TABLE IF NOT EXISTS Songs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name VARCHAR(100) NOT NULL,
        album_id INTEGER NOT NULL,
        genre VARCHAR(50),
        duration_seconds INTEGER,
        FOREIGN KEY (album_id) REFERENCES Albums(id)
    );
    ''')

    for song_name, artist_name, album_name, genre, duration in music_data:
        # 1. 查或插入 Artist
        cursor.execute("SELECT id FROM Artists WHERE name = ?", (artist_name,))
        res = cursor.fetchone()
        if res:
            artist_id = res[0]
        else:
            cursor.execute("INSERT INTO Artists (name) VALUES (?)", (artist_name,))
            artist_id = cursor.lastrowid

        # 2. 查或插入 Album (注意：同一歌手可能有同名专辑，这里简化处理)
        cursor.execute("SELECT id FROM Albums WHERE name = ? AND artist_id = ?", (album_name, artist_id))
        res = cursor.fetchone()
        if res:
            album_id = res[0]
        else:
            cursor.execute("INSERT INTO Albums (name, artist_id) VALUES (?, ?)", (album_name, artist_id))
            album_id = cursor.lastrowid

        # 3. 插入 Song
        cursor.execute(
            "INSERT INTO Songs (name, album_id, genre, duration_seconds) VALUES (?, ?, ?, ?)",
            (song_name, album_id, genre, duration)
        )

    conn.commit()
    conn.close()
    print("✅ 14首音乐已成功存入 music.db 数据库！")

if __name__ == "__main__":
    init_db_and_import()

✅ 14首音乐已成功存入 music.db 数据库！


In [11]:
import sqlite3
import random

db_path = "E:/LangChain/Chapter_2_Langchain_S/music.db"

# 精选的婚礼歌曲数据 (歌名, 歌手, 专辑, 流派标签, 时长秒数)
wedding_songs = [
    # 中文
    ("我愿意", "王菲", "菲靡靡之音", "婚礼-深情告白", 268),
    ("告白气球", "周杰伦", "周杰伦的床边故事", "婚礼-甜蜜浪漫", 215),
    ("今天你要嫁给我", "陶喆 & 蔡依林", "太美丽", "婚礼-欢快对唱", 272),
    ("最重要的决定", "范玮琪", "Love and FanFan", "婚礼-誓言时刻", 246),
    ("小幸运", "田馥甄", "我的少女时代 电影原声带", "婚礼-清新甜蜜", 265),
    ("慢慢喜欢你", "莫文蔚", "我们在中场相遇", "婚礼-温柔背景", 221),
    ("月亮代表我的心", "邓丽君", "岛国之情歌第四集", "婚礼-经典隽永", 206),
    ("往后余生", "马良", "往后余生", "婚礼-坚定承诺", 221),
    ("这世界那么多人", "莫文蔚", "这世界那么多人", "婚礼-温柔治愈", 285),
    ("终于等到你", "张靓颖", "第七感", "婚礼-契合主题", 292),
    # 英文
    ("Perfect", "Ed Sheeran", "÷ (Divide)", "Wedding-Romantic", 263),
    ("A Thousand Years", "Christina Perri", "The Twilight Saga: Breaking Dawn", "Wedding-Ceremony", 285),
    ("Canon in D", "Pachelbel", "Classical Wedding Music", "Wedding-Processional", 300),
    ("Marry You", "Bruno Mars", "Doo-Wops & Hooligans", "Wedding-Recessional", 230),
    ("Thinking Out Loud", "Ed Sheeran", "x (Multiply)", "Wedding-First Dance", 281),
    ("Beautiful In White", "Shane Filan", "Beautiful In White - Single", "Wedding-Vow", 233),
    ("Sugar", "Maroon 5", "V", "Wedding-Upbeat", 235),
    ("Can't Help Falling in Love", "Elvis Presley", "Blue Hawaii", "Wedding-Classic", 180),
    ("I Do", "911", "Moving On", "Wedding-Closing", 210),
    # 纯音乐
    ("River Flows in You", "Yiruma", "First Love", "Wedding-Instrumental", 188),
]

def insert_and_price_wedding_songs():
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("PRAGMA foreign_keys = ON;")

    # 1. 确保 price 字段存在
    try:
        cursor.execute("ALTER TABLE Songs ADD COLUMN price REAL DEFAULT 0.0;")
        print("✅ 成功添加 price 字段")
    except sqlite3.OperationalError:
        pass # 字段已存在

    inserted_count = 0
    updated_count = 0

    for song_name, artist_name, album_name, genre, duration in wedding_songs:
        # 生成一个模拟价格，例如 2.99 到 9.99 元
        mock_price = round(random.uniform(2.99, 9.99), 2)

        try:
            # 查或插 Artist
            cursor.execute("SELECT id FROM Artists WHERE name = ?", (artist_name,))
            res = cursor.fetchone()
            if res:
                artist_id = res[0]
            else:
                cursor.execute("INSERT INTO Artists (name) VALUES (?)", (artist_name,))
                artist_id = cursor.lastrowid

            # 查或插 Album
            cursor.execute("SELECT id FROM Albums WHERE name = ? AND artist_id = ?", (album_name, artist_id))
            res = cursor.fetchone()
            if res:
                album_id = res[0]
            else:
                cursor.execute("INSERT INTO Albums (name, artist_id) VALUES (?, ?)", (album_name, artist_id))
                album_id = cursor.lastrowid

            # 查或插 Song (并带上价格)
            cursor.execute("SELECT id FROM Songs WHERE name = ? AND album_id = ?", (song_name, album_id))
            song_res = cursor.fetchone()

            if song_res:
                # 如果歌曲已经存在，就更新它的价格
                song_id = song_res[0]
                cursor.execute("UPDATE Songs SET price = ? WHERE id = ?", (mock_price, song_id))
                updated_count += 1
            else:
                # 如果歌曲不存在，插入并带上价格
                cursor.execute(
                    "INSERT INTO Songs (name, album_id, genre, duration_seconds, price) VALUES (?, ?, ?, ?, ?)",
                    (song_name, album_id, genre, duration, mock_price)
                )
                inserted_count += 1

        except sqlite3.Error as e:
            print(f"处理 {song_name} 时出错: {e}")

    conn.commit()
    conn.close()
    print(f"🎉 操作完成！新插入了 {inserted_count} 首歌，更新了 {updated_count} 首歌的价格。")

if __name__ == "__main__":
    insert_and_price_wedding_songs()

✅ 成功添加 price 字段
🎉 操作完成！新插入了 0 首歌，更新了 20 首歌的价格。


#### 连接SQLite数据库

In [6]:
from langchain_community.utilities import SQLDatabase


db_uri = "sqlite:///E:/LangChain/Chapter_2_Langchain_S/music.db"
db = SQLDatabase.from_uri(db_uri)

#### 工具准备

In [7]:
@tool
def query_playlist(query: str) -> str:
    """Query the database for playlist information"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying the database: {e}"

In [8]:
playlist_agent = create_agent(
    model="deepseek-flash",
    tools=[query_playlist],
    system_prompt="""
    -你是播放列表专家。查询sql数据库。
    -一旦你有了你的播放列表，计算播放列表的总时长和成本，每首歌都有一个相关的价格。
    -如果在查询数据库时遇到错误，请尝试通过更改查询来修复它们。
    -不要空手而归，继续尝试查询数据库，直到你找到一个歌曲列表。
    -您可能需要进行多个查询以迭代地找到最佳选项。
"""
)

#### 测试

In [9]:
response = playlist_agent.invoke({"messages" : [HumanMessage("查询铁痕电台的歌曲")]})
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

查询铁痕电台的歌曲
================================== Ai Message ==================================
Tool Calls:
  query_playlist (call_00_lJX1Apd1bMlueWCS5Zys2900)
 Call ID: call_00_lJX1Apd1bMlueWCS5Zys2900
  Args:
    query: 查询铁痕电台的歌曲
================================= Tool Message =================================
Name: query_playlist

Error querying the database: (sqlite3.OperationalError) near "查询铁痕电台的歌曲": syntax error
[SQL: 查询铁痕电台的歌曲]
(Background on this error at: https://sqlalche.me/e/21/e3q8)
================================== Ai Message ==================================
Tool Calls:
  query_playlist (call_00_dtU0YEyMjkGCNU69TiP50325)
 Call ID: call_00_dtU0YEyMjkGCNU69TiP50325
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table'
================================= Tool Message =================================
Name: query_playlist

[('Artists',), ('sqlite_sequence',), ('Albums',), ('Songs',)]
=

### 主agent

In [9]:
class WeddingState(AgentState):
    origin: str
    destination: str
    guest_count: str
    genre: str

In [10]:
@tool
def update_state(origin: str, destination: str, guest_count: str, genre: str, runtime :ToolRuntime) -> str:
    """收集并更新婚礼规划所需的状态信息：出发地、目的地、宾客人数和音乐风格"""

    return Command(update={
                   "origin" : origin,
                   "destination" : destination,
                   "guest_count" : guest_count,
                   "genre" : genre,
                    "messages" : [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]
                   }
    )

@tool
async def search_flight(runtime: ToolRuntime) -> str:
    """旅行agent搜索飞往理想婚礼地点的航班"""

    origin = runtime.state.get("origin")
    destination = runtime.state.get("destination")

    content = f"从{origin}到{destination}的航班"
    response = await flight_agent.ainvoke({"messages" : [HumanMessage(content)]})
    return response["messages"][-1].content

@tool
async def search_venues(runtime: ToolRuntime) -> str:
    """旅行agent搜索理想婚礼地点"""

    destination = runtime.state.get("destination")
    capacity = runtime.state.get("guest_count")

    content = f"在{destination}寻找容纳{capacity}人的婚礼地点"
    response = await venue_agent.ainvoke({"messages" : [HumanMessage(content)]})
    return response["messages"][-1].content

@tool
async def search_music(runtime: ToolRuntime) -> str:
    """音乐agent搜索婚礼音乐"""

    genre = runtime.state.get("genre")

    content = f"策划符合{genre}风格的婚礼音乐"
    response = await playlist_agent.ainvoke({"messages" : [HumanMessage(content)]})
    return response["messages"][-1].content

In [11]:
main_agent = create_agent(
    model="deepseek-flash",
    tools=[update_state, search_flight, search_venues, search_music],
    state_schema=WeddingState,
    system_prompt="""
    你是婚礼协调员。你负责与用户沟通，找到更新状态所需的所有信息，并更新到state。
    一旦完成，你就可以委派任务了。将找航班、搜索场地和策划播放列表的任务委派给您的专家Agent。
    当你收到他们的答案，为我设计一场完美的婚礼方案。
"""
)

#### 测试

In [12]:
response = await main_agent.ainvoke({"messages" : [HumanMessage("我来自北京，我想在昆明办一个100人的婚礼，风格随意")]})

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

我来自北京，我想在昆明办一个100人的婚礼，风格随意
================================== Ai Message ==================================

好的！我来帮您筹备这场婚礼。首先，我先整理一下您提供的信息：

- **出发地**：北京
- **目的地**：昆明
- **宾客人数**：100人
- **婚礼风格**：随意（轻松自在的休闲风格）

我先把这些信息更新到系统中。
Tool Calls:
  update_state (call_00_qHd8Tr7ojZD4KORgMzUB9753)
 Call ID: call_00_qHd8Tr7ojZD4KORgMzUB9753
  Args:
    origin: 北京
    destination: 昆明
    guest_count: 100
    genre: 随意（轻松自在的休闲风格）
================================= Tool Message =================================
Name: update_state

Successfully updated state
================================== Ai Message ==================================

信息已更新完成！现在我将为您同时派出三位专家：旅行Agent负责搜索飞往昆明的航班和合适的婚礼场地，音乐Agent负责策划契合"随意"风格的婚礼播放列表。
Tool Calls:
  search_flight (call_00_WYzTunTKrNBj7SGGcI2d5535)
 Call ID: call_00_WYzTunTKrNBj7SGGcI2d5535
  Args:
  search_venues (call_01_qmfGS35BcRUi6oM7M6q40603)
 Call ID: call_01_qmfGS35BcRUi6oM7M6q40603
  Ar